# OVERTURE

# imports

In [4]:
import os
import torch
from torch import nn , optim 
from torch.utils.data import DataLoader,Dataset,random_split
from torchinfo import summary
from torchvision import datasets, transforms
from torchvision.transforms import Compose, ColorJitter, ToTensor
from torchvision.io import decode_image
from torchvision.datasets import ImageFolder
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from lightning.pytorch import Trainer
from PIL import Image
from datasets import load_dataset,Image
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Data Preprocessing

In [5]:
batch_size = 16

image_height = 224
image_width = 224

preprocess = transforms.Compose([
    transforms.Resize((image_width,image_height)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    # transforms.ColorJitter(brightness=0.3,contrast=0.3,saturation=0.3),
    transforms.ToTensor(), # Converts PIL to [3, 32, 32] Tensor -> rgb size 32*32
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

valPreprocess = transforms.Compose([
    transforms.Resize((image_width,image_height)),
    transforms.RandomRotation(15),
    transforms.ToTensor(), # Converts PIL to [3, 32, 32] Tensor -> rgb size 32*32
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])



finalTestPreprocess = transforms.Compose([
    transforms.Resize((image_height,image_width)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

#ImageFolder automatically one hot folder(no as 0,yes as 1) so we dont have to put label ourselves
brain_dataset =ImageFolder(root='brain_tumor_dataset', transform=None)

#test train val split -> split now so dataloader and transformation can be done with no problem
train_size = round(0.7*len(brain_dataset)) #split wont take float,we need to round that up to int
val_size = round(0.15*len(brain_dataset))
test_size = len(brain_dataset) - train_size - val_size
train_dataset, val_dataset , test_dataset = random_split(
    brain_dataset,
    [train_size, val_size,test_size],
    generator=torch.Generator().manual_seed(42)
    )

#wrapper class,so we can do separate transformation and have test unchanged
class BrainTumorDataset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
        
    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y
        
    def __len__(self):
        return len(self.subset)

train_dataset = BrainTumorDataset(subset=train_dataset, transform=preprocess)
val_dataset = BrainTumorDataset(subset=val_dataset, transform=valPreprocess)
test_dataset = BrainTumorDataset(subset=test_dataset, transform=finalTestPreprocess)

#put into dataloader for iterations in the model
train_loader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=batch_size,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=batch_size,shuffle=False)

#get amount of output classes
classes_amount = len(brain_dataset.classes)
print(classes_amount)

image , label = next(iter(train_loader))
print(image,label)

2
tensor([[[[-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          ...,
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000]],

         [[-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          ...,
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000]],

         [[-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000,

# main model

conv layers will detect patterns that corelates to a tumor cells in the brain,we use batch norm and maxpooling to reduce unneeded data

for final dense layers we use 256 then 128 channels to force the model to discards unneeded data and reduce overall noises

In [6]:
class CNN_tumor(nn.Module):
    def __init__(self,in_channels,out_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels,out_channels = 32,kernel_size = 3)
        # self.reConv1 = nn.LazyConv2d(32,kernel_size = 3)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        # self.reConv2 = nn.LazyConv2d( 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        # self.reConv3 = nn.LazyConv2d(128, 3, padding=1)
        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)
        # self.reConv4 = nn.LazyConv2d(256, 3, padding=1)
        # self.conv5 = nn.Conv2d(256, 512, 3, padding=1)
        # self.reConv5 = nn.Conv2d(512, 512, 3, padding=1)
        self.maxPool = nn.MaxPool2d(kernel_size=3)
        self.avgPool = nn.AdaptiveAvgPool2d((1,1))
        self.batchNorm1 = nn.BatchNorm2d(32)
        self.batchNorm2 = nn.BatchNorm2d(64)
        self.batchNorm3 = nn.BatchNorm2d(128)
        self.batchNorm4 = nn.BatchNorm2d(256)
        # self.batchNorm5 = nn.BatchNorm2d(512)
        self.flatten = nn.Flatten()
        self.denseIn = nn.LazyLinear(256)
        self.dense = nn.Linear(128, 256)
        self.denseOut = nn.Linear(256, out_classes)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax()
        self.dropout = nn.Dropout(0.5)
        
    def forward(self,x):
        residual = x
        x = self.relu(self.batchNorm1(self.conv1(x)))
        x = self.maxPool(x)
        # x += self.maxPool(x)
        x = self.relu(self.batchNorm2(self.conv2(x)))
        x = self.maxPool(x)
        x = self.relu(self.batchNorm3(self.conv3(x)))
        x = self.maxPool(x)
        x = self.relu(self.batchNorm4(self.conv4(x)))
        x = self.maxPool(x)
        # x+= self.maxPool(self.maxPool(self.maxPool(self.maxPool(residual))))
        # x = self.relu(self.batchNorm5(self.conv5(x)))
        # x = self.maxPool(x)
        x = self.avgPool(x)
        x = self.flatten(x)
        x = self.relu(self.denseIn(x))
        x = self.dropout(x)
        return self.denseOut(x)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = CNN_tumor(in_channels=3,out_classes=classes_amount).to(device)

# loss func set
criterion = nn.CrossEntropyLoss()
# adam optim , learning rate 0.001
optimizer = optim.Adam(model.parameters(), lr=0.001) 

print(model)
summary(model, input_size=(100,3,image_height,image_width))# (Batch Size, Channels, Height, Width)


CNN_tumor(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (maxPool): MaxPool2d(kernel_size=3, stride=3, padding=0, dilation=1, ceil_mode=False)
  (avgPool): AdaptiveAvgPool2d(output_size=(1, 1))
  (batchNorm1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (batchNorm2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (batchNorm3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (batchNorm4): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (denseIn): LazyLinear(in_features=0, out_features=256, bias=True)
  (dense): Linear(in_features=128, out_features=256, bia

Layer (type:depth-idx)                   Output Shape              Param #
CNN_tumor                                [100, 2]                  33,024
├─Conv2d: 1-1                            [100, 32, 222, 222]       896
├─BatchNorm2d: 1-2                       [100, 32, 222, 222]       64
├─ReLU: 1-3                              [100, 32, 222, 222]       --
├─MaxPool2d: 1-4                         [100, 32, 74, 74]         --
├─Conv2d: 1-5                            [100, 64, 74, 74]         18,496
├─BatchNorm2d: 1-6                       [100, 64, 74, 74]         128
├─ReLU: 1-7                              [100, 64, 74, 74]         --
├─MaxPool2d: 1-8                         [100, 64, 24, 24]         --
├─Conv2d: 1-9                            [100, 128, 24, 24]        73,856
├─BatchNorm2d: 1-10                      [100, 128, 24, 24]        256
├─ReLU: 1-11                             [100, 128, 24, 24]        --
├─MaxPool2d: 1-12                        [100, 128, 8, 8]          --


# Training and eval

In [7]:
epochs = 30

# For tracking progress
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

early_stop_callback = EarlyStopping(monitor="val_loss", min_delta=0.00, patience=3, verbose=False, mode="min")
trainer = Trainer(callbacks=[early_stop_callback])


for epoch in range(epochs):
    # --- TRAINING PHASE ---
    model.train()
    train_loss, train_correct = 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # 1. Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # 2. Backward pass (The math magic)
        optimizer.zero_grad() # Clear old gradients
        loss.backward()       # Calculate new gradients
        optimizer.step()      # Update weights
        
        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        train_correct += (predicted == labels).sum().item()

    # --- VALIDATION PHASE ---
    model.eval()
    val_loss, val_correct = 0, 0
    
    with torch.no_grad(): # Don't calculate gradients here (saves memory)
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()

    # Calculate averages
    train_loss /= len(train_loader.dataset)
    train_acc = train_correct / len(train_loader.dataset)
    val_loss /= len(val_loader.dataset)
    val_acc = val_correct / len(val_loader.dataset)
    
    # Save to history
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")
    

#final test
model.eval()
test_loss , test_correct = 0,0

with torch.no_grad(): # Don't calculate gradients here (saves memory)
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        test_correct += (predicted == labels).sum().item()
# Calculate averages
test_loss /= len(test_loader.dataset)
test_acc = test_correct / len(test_loader.dataset)
# val_loss /= len(val_loader.dataset)
# val_acc = val_correct / len(val_loader.dataset)
print(f"final test =>  Model loss : {test_loss:.4f} | Test accuracy : {test_acc:.4f}")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Epoch [1/30] | Train Loss: 0.6626 | Val Acc: 0.6579
Epoch [2/30] | Train Loss: 0.5373 | Val Acc: 0.6579
Epoch [3/30] | Train Loss: 0.4798 | Val Acc: 0.6842
Epoch [4/30] | Train Loss: 0.5192 | Val Acc: 0.7632
Epoch [5/30] | Train Loss: 0.4442 | Val Acc: 0.7632
Epoch [6/30] | Train Loss: 0.4612 | Val Acc: 0.6579
Epoch [7/30] | Train Loss: 0.3806 | Val Acc: 0.8421
Epoch [8/30] | Train Loss: 0.4359 | Val Acc: 0.6579
Epoch [9/30] | Train Loss: 0.3795 | Val Acc: 0.8158
Epoch [10/30] | Train Loss: 0.3723 | Val Acc: 0.7105
Epoch [11/30] | Train Loss: 0.4881 | Val Acc: 0.8421
Epoch [12/30] | Train Loss: 0.3447 | Val Acc: 0.7632
Epoch [13/30] | Train Loss: 0.3136 | Val Acc: 0.6053
Epoch [14/30] | Train Loss: 0.4199 | Val Acc: 0.4211
Epoch [15/30] | Train Loss: 0.3998 | Val Acc: 0.7632
Epoch [16/30] | Train Loss: 0.3979 | Val Acc: 0.7895
Epoch [17/30] | Train Loss: 0.3310 | Val Acc: 0.7632
Epoch [18/30] | Train Loss: 0.3045 | Val Acc: 0.7632
Epoch [19/30] | Train Loss: 0.2619 | Val Acc: 0.7632
Ep

# Outside Test

# Model Save

In [ ]:
PATH = "BrainTumorModel.pth"
torch.save(model.state_dict(), PATH)